# Your SVD algorithm → metrics

Write `my_svd(a, rank)` in the cell below and *Run All*. The notebook runs it on a fixed
set of matrices, compares it with an FP64 reference SVD and with `torch.linalg.svd`, and
reports accuracy and time. On a GPU host it also checks the method on the GPU.

**Function contract**
- `a`: a `torch.Tensor` of shape `(m, n)` (float32/float64/complex) on `DEVICE`;
- `rank`: how many singular triplets are needed (`rank == min(m, n)` for a full SVD);
- return `(U, s, Vh)` with shapes `(m, k)`, `(k,)`, `(k, n)` and `k >= rank` (extra triplets
  are truncated); NumPy arrays work, and returning only `s` checks the spectrum alone.

Same from a terminal: put `my_svd` in a file and run `python -m svdlab check my_svd.py:my_svd` (see `examples/`).

In [ ]:
from pathlib import Path
import sys, importlib
import numpy as np
import torch

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "svdbench").is_dir():
        ROOT = candidate
        sys.path.insert(0, str(ROOT))
        break
else:
    raise RuntimeError("Open this notebook from the Pavel-s_SVD repository")

from svdlab import scorecard
importlib.reload(scorecard)
from svdlab.scorecard import evaluate, compare, gpu_check, table, summary, DEFAULT_SUITE
from svdbench.laboratory import MatrixCase

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
DEVICE

## 1. Your algorithm

The starter below is a one-sided (Hestenes) Jacobi SVD in plain torch. Replace the body
with your method; `breakpoint()` inside it works too.

In [ ]:
def my_svd(a, rank):
    """One-sided Jacobi: orthogonalize columns of W = A V by plane rotations."""
    m, n = a.shape
    transpose = m < n
    w = (a.mH if transpose else a).clone()
    v = torch.eye(w.shape[1], dtype=w.dtype, device=w.device)
    eps = torch.finfo(w.real.dtype if w.is_complex() else w.dtype).eps
    for sweep in range(30):
        rotated = False
        for p in range(w.shape[1] - 1):
            for q in range(p + 1, w.shape[1]):
                alpha = (w[:, p].abs() ** 2).sum().real
                beta = (w[:, q].abs() ** 2).sum().real
                gamma = (w[:, p].conj() @ w[:, q])
                if gamma.abs() <= eps * torch.sqrt(alpha * beta):
                    continue
                rotated = True
                phase = gamma / gamma.abs()
                zeta = (beta - alpha) / (2 * gamma.abs())
                t = torch.sign(zeta) / (zeta.abs() + torch.sqrt(1 + zeta ** 2))
                t = torch.where(zeta == 0, torch.ones_like(t), t)
                c = 1 / torch.sqrt(1 + t ** 2); s = c * t
                for x in (w, v):
                    xp, xq = x[:, p].clone(), x[:, q].clone()
                    x[:, p] = c * xp - s * phase.conj() * xq
                    x[:, q] = s * phase * xp + c * xq
        if not rotated:
            break
    sigma = torch.linalg.vector_norm(w, dim=0)
    order = torch.argsort(sigma, descending=True)
    sigma, w, v = sigma[order], w[:, order], v[:, order]
    u = w / torch.where(sigma > 0, sigma, torch.ones_like(sigma))
    u, sigma, vh = u[:, :rank], sigma[:rank], v[:, :rank].mH
    return (vh.mH, sigma, u.mH) if transpose else (u, sigma, vh)

## 2. Quick check on small matrices

Python-level Jacobi loops are slow, so start with tiny matrices.
`MatrixCase.create(...)` accepts the families `gaussian, low_rank, clustered, geometric, sparse, zero`;
`MatrixCase.load("matrix.npy")` loads your own matrix and `MatrixCase.from_spectrum([...], m=, n=)`
builds one with a chosen spectrum.

In [ ]:
small = [dict(name="gaussian_24x16", m=24, n=16, family="gaussian", dtype="float64"),
         dict(name="complex_16x24", m=16, n=24, family="gaussian", dtype="complex128"),
         dict(name="low_rank_k4", m=30, n=20, family="low_rank", rank=4, tail=1e-6, dtype="float64", output_rank=4)]
rows = evaluate(my_svd, small, device=DEVICE, repeat=3)
table(rows)

## 3. Compare with builtin methods

A method is a function or a name: `"torch"`, `"numpy"`, `"mine"` (`my_algorithm.py`),
`"lab.randomized"`, `"lab.jacobi"`, or any of the 16 `svdbench.standalone` variants:
`bidiag_qr, bidiag_dc, jacobi, block_jacobi, qr, tsqr, gram_jacobi, gram_dc, polar_qdwh,
polar_newton_schulz, polar_halley, polar_polynomial, randomized, subspace, block_krylov, lanczos`.

The standard suite (`DEFAULT_SUITE`) has 8 matrices up to 400×300. For slow Python code,
shrink it or keep using `small`.

In [ ]:
print([c["name"] for c in DEFAULT_SUITE])
CASES = small                     # DEFAULT_SUITE once your method is fast enough
rows = compare({"my_svd": my_svd, "torch": "torch", "bidiag_qr": "bidiag_qr", "polar_qdwh": "polar_qdwh"},
               CASES, device=DEVICE, repeat=5)
df = table(rows)
df

In [ ]:
import pandas as pd
display(pd.DataFrame(summary(rows)))
df.pivot(index="case", columns="method", values="time_median_ms")

### What the columns mean

| column | meaning |
|---|---|
| `ok` | U/V orthogonality and reconstruction ≤ 100·eps·max(m,n); for a truncated SVD, residual ≤ 1.05× the optimal (Eckart–Young) tail |
| `time_median_ms`, `time_mad_ms` | median and MAD of the call alone (no matrix generation or validation) |
| `device_time_median_ms`, `gpu_peak_increment_bytes` | GPU only: CUDA-event time and Torch allocator peak above baseline |
| `output_device` | where the returned arrays live; a GPU run returning `cpu` or `numpy` computed on the host |
| `reconstruction_relative_fro` | ‖A − U Σ Vᴴ‖_F / ‖A‖_F |
| `residual_over_optimal_tail` | residual / ‖σ_{k+1:}‖: how much worse than the best rank-k approximation |
| `resolved_spectrum_max_relative_error` | max relative error of σᵢ above the FP64 noise floor |
| `spectrum_error_over_sigma1` | max \|σᵢ − σᵢ_ref\| / σ₁ |
| `orthogonality_u/v_...` | ‖UᴴU − I‖_F / √k, and the same for V |
| `left/right_triplet_relative_fro` | ‖A vᵢ − σᵢ uᵢ‖ and ‖Aᴴ uᵢ − σᵢ vᵢ‖ relative to ‖A‖ |
| `right_subspace_leakage_...` | right subspace outside the reference one (only with a spectral gap) |
| `singular_values_nonnegative_sorted` | σ ≥ 0 and sorted in descending order |

## 4. Plots and your own matrices

In [ ]:
import matplotlib.pyplot as plt
mine = [r for r in rows if r["method"] == "my_svd" and r["status"] == "done"]
ref = {r["case"]: r for r in rows if r["method"] == "torch"}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
names = [r["case"] for r in mine]
ax[0].bar(names, [r["reconstruction_relative_fro"] for r in mine], label="my_svd")
ax[0].scatter(names, [ref[c]["reconstruction_relative_fro"] for c in names], color="k", zorder=3, label="torch")
ax[0].set_yscale("log"); ax[0].set_title("‖A − UΣVᴴ‖ / ‖A‖"); ax[0].legend()
ax[1].bar(names, [r["time_median_ms"] for r in mine], label="my_svd")
ax[1].scatter(names, [ref[c]["time_median_ms"] for c in names], color="k", zorder=3, label="torch")
ax[1].set_yscale("log"); ax[1].set_title("median time, ms"); ax[1].legend()
for x in ax: x.tick_params(axis="x", rotation=30)
plt.tight_layout()

In [ ]:
# Your own matrix: a chosen spectrum with a repeated value, or a .npy from the broad benchmark
custom = [MatrixCase.from_spectrum([5, 5, 1, 1e-3, 1e-9, 0], m=10, n=8, seed=1)]
# custom = [MatrixCase.load("experiments/broad-20261012/lowrank2048x1024_k32/matrix.npy")]
table(evaluate(my_svd, custom, device=DEVICE, repeat=3))

In [ ]:
# Save the results
scorecard.save(rows, ROOT / "laboratory_runs" / "scorecard.csv")

## 5. GPU check

Runs the same method on CPU and on the GPU and passes a case only when the outputs come
back on the GPU, both runs are `ok`, and the singular values agree within the case tolerance.
This catches a GPU path that breaks numerically or silently falls back to the host.
Skipped on a machine without CUDA/ROCm. Terminal equivalent:
`python -m svdlab check my_svd.py:my_svd --gpu-check`.

In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    gpu_rows = gpu_check(my_svd, CASES, device="cuda:0", repeat=5, verbose=False)
    display(pd.DataFrame(gpu_rows))
    assert all(r["passed"] for r in gpu_rows), "GPU check failed; see the table above"
else:
    print("No CUDA/ROCm device visible to torch; GPU check skipped")

## Next steps

- Step-by-step tracing of internal iterations (snapshots of Q, B and errors per iteration):
  `python -m algorithm_laboratory.experiment --algorithm mine`; see `laboratory.ipynb`.
- GPU on Hetzner: same notebook or CLI with the CUDA Python `/root/svdbench/.venv/bin/python`
  (see `docs/ALGORITHM_LABORATORY.md`).